# Experiment 2: attack-rate gradient

Run the shared AUTO model and save this experiment's figures.

In [ ]:
import os
from pathlib import Path

from AUTOclui import AUTOCommands as ac
from AUTOclui import runAUTO as ra
from pyvirtualdisplay import Display

In [ ]:
folder = Path('/auto/Simulations')
os.chdir(folder)

model_name = 'common_model'
output_folder = folder / 'output_experiment_two_attack_rate'
output_folder.mkdir(exist_ok=True)

parameter_file = folder / 'experiment_parameters.dat'
parameter_file.write_text('0.0 0.0\n')

In [ ]:
display = Display(visible=False, size=(1200, 900))
display.start()

In [ ]:
runner = ra.runAUTO()

try:
    eq_forward = ac.run(e=model_name, c=model_name, runner=runner, NMX=4000, NPR=200)
    eq_backward = ac.run(DS='-', runner=runner, NMX=4000, NPR=200)
    eq = (eq_forward + eq_backward).relabel()
    ac.save(eq, 'eq')

    bp_curve = ac.run(
        eq('BP1'),
        ICP=[28, 30],
        ISW=2,
        DS=1.0e-3,
        DSMIN=1.0e-5,
        DSMAX=5.0e-3,
        NMX=8000,
        NPR=400,
        UZSTOP={28: [-0.25, 0.5], 30: [0.0, 0.4]},
        runner=runner,
    ).relabel()
    ac.save(bp_curve, 'bp_curve')

    lp_curve = ac.run(
        eq('LP1'),
        ICP=[28, 30],
        ISW=2,
        DS=1.0e-3,
        DSMIN=1.0e-5,
        DSMAX=5.0e-3,
        NMX=8000,
        NPR=400,
        UZSTOP={28: [-0.25, 0.5], 30: [0.0, 0.4]},
        runner=runner,
    ).relabel()
    ac.save(lp_curve, 'lp_curve')

    codim2 = (bp_curve + lp_curve).relabel()
    ac.save(codim2, 'codim2')
finally:
    runner.config(clean=True)
    ac.clean()

In [ ]:
p = ac.plot('eq', hide=True)
p.config(
    stability=True,
    grid=False,
    bifurcation_x=['mu'],
    bifurcation_y=['PL'],
    xlabel='mu',
    ylabel='PL',
    title='',
    minx=0.0,
    maxx=0.1,
)
p.savefig(str(output_folder / 'experiment_two_attack_rate_1d.png'))
p.savefig(str(output_folder / 'experiment_two_attack_rate_1d.svg'))

In [ ]:
p = ac.plot('codim2', hide=True)
p.config(
    grid=False,
    bifurcation_x=['deltas'],
    bifurcation_y=['mu'],
    xlabel='deltas',
    ylabel='mu',
    title='',
    minx=0.0,
    maxx=0.4,
    miny=0.0,
    maxy=0.1,
)
p.savefig(str(output_folder / 'experiment_two_attack_rate_2d.png'))
p.savefig(str(output_folder / 'experiment_two_attack_rate_2d.svg'))

In [ ]:
display.stop()
parameter_file.unlink(missing_ok=True)
ac.delete('eq')
ac.delete('bp_curve')
ac.delete('lp_curve')
ac.delete('codim2')